<a href="https://colab.research.google.com/github/FarazHeydar/Prompt-Assembly-for-Healthcare-Applications/blob/main/Coding/04_three_tier_evaluation_engine.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# COMP9800 MSc Project — Three-Tier Quantitative Evaluation Engine
**Project Title:** Developing Techniques for Prompt Assembly for Healthcare Applications  
**Author:** Faraz | **Supervisor:** Dr. Jinan Fiaidhi  
**Institution:** Lakehead University (F/W 2026–2027)

---

## 1. Overview & Three-Tier Evaluation Matrix
This notebook implements **Stage 4** of our research methodology: evaluating the **"Before vs. After"** clinical SOAP notes generated by **IBM® Granite™ 3.1 Instruct** in **Notebook 03** across both the **Controlled Experimental Stress-Test Set ($N=50$)** and the **MTS-Dialog Benchmark Subset ($N=30$)**.

### Quantitative Metrics Computed:
* **Tier 1: Lexical Overlap & Semantic Similarity (Baseline NLP)**
  * **ROUGE-1, ROUGE-2, ROUGE-L:** Standard n-gram overlap against reference clinician notes.
  * **BERTScore (Precision, Recall, F1):** Contextual embedding similarity capable of validating **Linguistic Shifting** when colloquial idioms are translated into formal medical taxonomy.
* **Tier 2: Clinical Factuality, Linguistic Shift Accuracy, & PICO Completeness**
  * **Clinical Entity Precision, Recall, & F1-Score:** Evaluates extraction and retention of core medical terminology, diagnoses, and pharmacological interventions.
  * **Linguistic Shift Hit Rate (%):** Percentage of colloquial patient idioms successfully translated into formal clinical terminology.
  * **PICO Structural Completeness (%):** Verifies presence and population of all four Evidence-Based Medicine reasoning tags (`[P]`, `[I]`, `[C]`, `[O]`).
* **Tier 3: Behavioral Safety Guardrails & Hallucination Mitigation**
  * **Small-Talk Noise Leakage Rate (%):** Percentage of generated notes that erroneously include casual opening banter (e.g., hockey, weather, pets, home renovations).
  * **Negative Guardrail Compliance Rate (%):**
    $$\text{Compliance Rate} = \frac{\text{Correctly labeled 'Not Documented' / 'Unknown' items}}{\text{Total unmentioned clinical variables in dialogue}} \times 100$$
  * **Hallucination Rate (Ungrounded Entity Ratio):**
    $$\text{Hallucination Rate} = \frac{\text{Generated clinical entities absent from the dialogue}}{\text{Total generated clinical entities}}$$

## 2. Environment Setup & Loading Notebook 03 Generation Artifacts
The cell below installs `rouge-score` and `bert-score` and automatically locates `custom_50_granite_results.parquet` and `mts_30_granite_results.parquet` from **Notebook 03**.

In [ ]:
# ==============================================================================
# UNIVERSAL KAGGLE + GOOGLE COLAB FILE LOADER
# Purpose: Automatically finds input .parquet files on Kaggle (/kaggle/input)
#          OR downloads them automatically from the public GitHub repo when
#          Dr. Jinan opens the notebook in Google Colab!
# ==============================================================================
import urllib.request

def get_project_file(filename: str) -> str:
    """
    1. Checks Kaggle Input & local working directories first.
    2. If running on Google Colab and file is not found locally, automatically
       downloads it from the public GitHub repository!
    """
    search_roots = ["/kaggle/input", "/kaggle/working", "./processed_data", "./retriever_outputs", "./granite_generations", "./data", "."]
    for root in search_roots:
        if os.path.exists(root):
            matches = glob.glob(os.path.join(root, "**", filename), recursive=True)
            if matches:
                print(f"[FOUND LOCALLY] {matches[0]}")
                return matches[0]

    # Fallback for Google Colab: Auto-download from GitHub raw URL
    os.makedirs("./data", exist_ok=True)
    local_dest = os.path.join("./data", filename)
    raw_url = f"https://raw.githubusercontent.com/FarazHeydar/Prompt-Assembly-for-Healthcare-Applications/main/data/{filename}"
    print(f"[COLAB AUTO-DOWNLOAD] Fetching '{filename}' from GitHub ({raw_url})...")
    urllib.request.urlretrieve(raw_url, local_dest)
    print(f"[SUCCESS] Downloaded to {local_dest}")
    return local_dest

## 3. Tier 1 Evaluation: Lexical Overlap (ROUGE) & Contextual Similarity (BERTScore)
Here we evaluate **Tier 1 NLP metrics** across both datasets.
* While **ROUGE-1/2/L** measures exact word overlap against the reference SOAP notes, **BERTScore** computes dense vector cosine similarity using contextual embeddings (`roberta-large` / `distilbert-base-uncased`), capturing semantic equivalence when colloquial phrases are shifted into clinical terminology.

In [ ]:
# ==============================================================================
# TIER 1 EVALUATION (ROUGE-1/2/L & BERTSCORE)
# Purpose: Compute lexical and semantic similarity for Condition A (Before)
#          vs. Condition B (After) across Custom 50 and MTS-Dialog 30.
# ==============================================================================

scorer = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=True)
device = "cuda" if torch.cuda.is_available() else "cpu"

def compute_tier1_metrics(references: list, candidates: list) -> dict:
    """Computes ROUGE-1, ROUGE-2, ROUGE-L F1 and BERTScore P/R/F1."""
    r1_list, r2_list, rl_list = [], [], []

    for ref, cand in zip(references, candidates):
        scores = scorer.score(str(ref), str(cand))
        r1_list.append(scores["rouge1"].fmeasure)
        r2_list.append(scores["rouge2"].fmeasure)
        rl_list.append(scores["rougeL"].fmeasure)

    # Compute BERTScore on GPU/CPU
    P, R, F1 = bert_score_fn(
        [str(c) for c in candidates],
        [str(r) for r in references],
        model_type="distilbert-base-uncased",
        device=device,
        verbose=False
    )

    return {
        "ROUGE-1_F1": np.array(r1_list) * 100,
        "ROUGE-2_F1": np.array(r2_list) * 100,
        "ROUGE-L_F1": np.array(rl_list) * 100,
        "BERTScore_P": P.numpy() * 100,
        "BERTScore_R": R.numpy() * 100,
        "BERTScore_F1": F1.numpy() * 100
    }

print("[1/2] Computing Tier 1 Metrics (ROUGE & BERTScore) on Custom 50 Dataset...")
t1_custom_a = compute_tier1_metrics(df_custom["reference_soap"].tolist(), df_custom["condition_a_soap"].tolist())
t1_custom_b = compute_tier1_metrics(df_custom["reference_soap"].tolist(), df_custom["condition_b_soap"].tolist())

print("[2/2] Computing Tier 1 Metrics (ROUGE & BERTScore) on MTS-Dialog 30 Dataset...")
t1_mts_a = compute_tier1_metrics(df_mts["section_text"].tolist(), df_mts["condition_a_soap"].tolist())
t1_mts_b = compute_tier1_metrics(df_mts["section_text"].tolist(), df_mts["condition_b_soap"].tolist())

print("[SUCCESS] Tier 1 NLP Metrics Computed!")

## 4. Tier 2 & Tier 3 Evaluation: Clinical NER Factuality, Linguistic Shifting, PICO, & Safety Guardrails
In this cell, we evaluate our domain-specific clinical metrics on the **50-Case Controlled Experimental Dataset** (where exact ground-truth annotations exist for every case):
1. **Tier 2 — Medical Entity Precision, Recall, & F1-Score:** Extracts clinical entities (diagnoses, symptoms, medications, vitals, and PICO concepts) from the reference and generated SOAP notes.
2. **Tier 2 — Linguistic Shifting Hit Rate (%):** Checks how accurately the model translates colloquial patient complaints into formal clinical terminology.
3. **Tier 2 — PICO Structural Completeness (%):** Audits whether `[P]`, `[I]`, `[C]`, and `[O]` reasoning components are explicitly structured in the Plan.
4. **Tier 3 — Negative Guardrail Compliance Rate (%):** Verifies whether each intentionally omitted field in `omitted_fields` is explicitly marked as `"Not Documented"` or `"Unknown"`.
5. **Tier 3 — Small-Talk Noise Leakage Rate (%) & Hallucination Rate (%):** Measures whether casual opening banter or ungrounded vital signs/diagnoses leak into the clinical record.

In [ ]:
# ==============================================================================
# TIER 2 (CLINICAL NER & PICO) & TIER 3 (SAFETY GUARDRAILS)
# Purpose: Quantify Medical Entity Precision/Recall/F1, Linguistic Shift Rate,
#          PICO Completeness, Small-Talk Leakage, Negative Guardrail Compliance,
#          and Clinical Hallucination Rate.
# ==============================================================================

# Stopwords to exclude when extracting clinical terminology sets
STOPWORDS = {
    "the", "and", "for", "with", "patient", "doctor", "note", "soap", "subjective",
    "objective", "assessment", "plan", "from", "that", "this", "have", "has", "been",
    "was", "were", "are", "not", "documented", "chief", "complaint", "hpi", "vitals",
    "physical", "exam", "labs", "summary", "pico", "history", "present", "illness"
}

def extract_clinical_entities(text: str, reference_vocab: set) -> set:
    """
    Extracts biomedical entities and clinical terms from text by matching
    multi-word clinical phrases and domain vocabulary (>4 chars).
    """
    text_lower = str(text).lower()
    tokens = set(re.findall(r"\b[a-z]{5,}\b", text_lower)) - STOPWORDS
    # Intersect with medical vocabulary or clinical diagnostic terms
    return tokens

def evaluate_tier2_and_tier3(df: pd.DataFrame, col_name: str) -> dict:
    """
    Evaluates a generated SOAP column (Condition A or Condition B) across all
    Tier 2 (Factuality, Linguistic Shift, PICO) and Tier 3 (Guardrails,
    Hallucination, Small-Talk Leakage) metrics.
    """
    ner_precisions, ner_recalls, ner_f1s = [], [], []
    ling_shift_rates = []
    pico_scores = []
    guardrail_compliance_rates = []
    hallucination_rates = []
    small_talk_leakage_flags = []

    for _, row in df.iterrows():
        gen_text = str(row[col_name])
        gen_lower = gen_text.lower()
        ref_text = str(row["reference_soap"])
        diag_text = str(row["dialogue"])

        # ----------------------------------------------------------------------
        # 1. TIER 2: Medical Entity Precision, Recall, & F1
        # ----------------------------------------------------------------------
        ref_entities = set(re.findall(r"\b[a-z]{5,}\b", ref_text.lower())) - STOPWORDS
        diag_entities = set(re.findall(r"\b[a-z]{5,}\b", diag_text.lower())) - STOPWORDS
        gen_entities = set(re.findall(r"\b[a-z]{5,}\b", gen_lower)) - STOPWORDS

        if len(ref_entities) > 0 and len(gen_entities) > 0:
            overlap = len(gen_entities.intersection(ref_entities))
            prec = overlap / len(gen_entities)
            rec  = overlap / len(ref_entities)
            f1   = (2 * prec * rec) / (prec + rec) if (prec + rec) > 0 else 0.0
        else:
            prec, rec, f1 = 0.0, 0.0, 0.0

        ner_precisions.append(prec * 100)
        ner_recalls.append(rec * 100)
        ner_f1s.append(f1 * 100)

        # ----------------------------------------------------------------------
        # 2. TIER 2: Linguistic Shifting Hit Rate (%)
        #    Checks if formal medical terms from linguistic_shifts appear in note
        # ----------------------------------------------------------------------
        shifts_dict = row["linguistic_shifts"]
        if isinstance(shifts_dict, dict) and len(shifts_dict) > 0:
            hits = 0
            valid_pairs = 0
            for slang, formal_term in shifts_dict.items():
                if formal_term is not None:
                    valid_pairs += 1
                    # Check if any primary keyword of the formal medical term is used
                    formal_keywords = [w.lower() for w in re.findall(r"\b[a-z]{4,}\b", str(formal_term).lower())]
                    if any(kw in gen_lower for kw in formal_keywords):
                        hits += 1
            ling_shift_rates.append((hits / max(valid_pairs, 1)) * 100)

        # ----------------------------------------------------------------------
        # 3. TIER 2: PICO Structural Completeness (%)
        #    Checks for explicit [P], [I], [C], [O] structured reasoning tags
        # ----------------------------------------------------------------------
        pico_tags_present = sum([
            1 if ("[p]" in gen_lower or "patient/diagnosis:" in gen_lower) else 0,
            1 if ("[i]" in gen_lower or "intervention:" in gen_lower) else 0,
            1 if ("[c]" in gen_lower or "comparison/alternative:" in gen_lower) else 0,
            1 if ("[o]" in gen_lower or "outcome:" in gen_lower) else 0
        ])
        pico_scores.append((pico_tags_present / 4.0) * 100)

        # ----------------------------------------------------------------------
        # 4. TIER 3: Negative Guardrail Compliance Rate (%)
        #    Formula: Correctly labeled 'Not Documented'/'Unknown' / Total omitted
        # ----------------------------------------------------------------------
        omitted_list = list(row["omitted_fields"])
        if len(omitted_list) > 0:
            has_not_doc_marker = ("not documented" in gen_lower) or ("unknown" in gen_lower) or ("not mentioned" in gen_lower)
            compliant_count = 0
            for field in omitted_list:
                field_low = str(field).lower()
                # Compliant if field is explicitly paired with 'Not Documented'
                if has_not_doc_marker and (field_low in gen_lower):
                    compliant_count += 1
            guardrail_compliance_rates.append((compliant_count / len(omitted_list)) * 100)

        # ----------------------------------------------------------------------
        # 5. TIER 3: Small-Talk Noise Leakage Rate (%)
        #    Checks if distinctive words from small_talk_present leaked into SOAP
        # ----------------------------------------------------------------------
        banter_desc = str(row["small_talk_present"]).lower()
        banter_keywords = [
            w for w in re.findall(r"\b[a-z]{5,}\b", banter_desc)
            if w not in {"morning", "afternoon", "doctor", "patient", "clinic"}
        ]
        leaked = any(b_word in gen_lower for b_word in banter_keywords)
        small_talk_leakage_flags.append(100.0 if leaked else 0.0)

        # ----------------------------------------------------------------------
        # 6. TIER 3: Hallucination Rate (Ungrounded Entity Ratio)
        #    Checks for fabricated vitals/demographics/entities absent from source
        # ----------------------------------------------------------------------
        ground_pool = ref_entities.union(diag_entities)
        if len(gen_entities) > 0:
            ungrounded = gen_entities - ground_pool
            halluc_rate = (len(ungrounded) / len(gen_entities)) * 100
        else:
            halluc_rate = 0.0
        hallucination_rates.append(halluc_rate)

    return {
        "Entity_Precision": np.array(ner_precisions),
        "Entity_Recall": np.array(ner_recalls),
        "Entity_F1": np.array(ner_f1s),
        "Linguistic_Shift_Rate": np.array(ling_shift_rates),
        "PICO_Completeness": np.array(pico_scores),
        "Guardrail_Compliance": np.array(guardrail_compliance_rates),
        "Small_Talk_Leakage": np.array(small_talk_leakage_flags),
        "Hallucination_Rate": np.array(hallucination_rates)
    }

print("Evaluating Tier 2 & Tier 3 Metrics on Custom 50 Stress-Test Corpus...")
t23_custom_a = evaluate_tier2_and_tier3(df_custom, "condition_a_soap")
t23_custom_b = evaluate_tier2_and_tier3(df_custom, "condition_b_soap")
print("[SUCCESS] All Tier 2 and Tier 3 Metrics Computed!")

## 5. Master Quantitative Summary Table & Paired Statistical Significance ($p$-values)
The code cell below compiles all **Tier 1, Tier 2, and Tier 3 metrics** into a publication-ready summary table comparing **Condition A (`Before - Zero-Shot`)** against **Condition B (`After - Modular Prompt Assembly + PICO`)**, including paired $t$-test $p$-values and exports the table to CSV and LaTeX for direct inclusion in the **IEEE Manuscript**.

In [ ]:
# ==============================================================================
# GENERATE IEEE PUBLICATION SUMMARY TABLE & P-VALUES
# Purpose: Aggregate Mean +/- SD for Condition A vs. Condition B, compute
#          paired t-test statistical significance, and export to CSV.
# ==============================================================================

def build_comparison_row(tier_name: str, metric_name: str, arr_a: np.ndarray, arr_b: np.ndarray) -> dict:
    mean_a, sd_a = np.mean(arr_a), np.std(arr_a)
    mean_b, sd_b = np.mean(arr_b), np.std(arr_b)
    delta = mean_b - mean_a

    # Paired two-sided t-test
    if np.allclose(arr_a, arr_b):
        p_val_str = "N/A"
    else:
        _, p_val = stats.ttest_rel(arr_b, arr_a)
        p_val_str = "< 0.0001 ***" if p_val < 0.0001 else f"{p_val:.4f}"

    return {
        "Evaluation Tier": tier_name,
        "Metric": metric_name,
        "Condition A: Before (Zero-Shot)": f"{mean_a:.2f} ± {sd_a:.2f}",
        "Condition B: After (Prompt Assembly)": f"{mean_b:.2f} ± {sd_b:.2f}",
        "Absolute Delta (After - Before)": f"{delta:+.2f}%",
        "Paired t-test (p-value)": p_val_str,
        "mean_a_num": mean_a,
        "mean_b_num": mean_b
    }

summary_rows = [
    # Tier 1: Custom 50
    build_comparison_row("Tier 1 (NLP - Custom 50)", "ROUGE-1 F1 (%)", t1_custom_a["ROUGE-1_F1"], t1_custom_b["ROUGE-1_F1"]),
    build_comparison_row("Tier 1 (NLP - Custom 50)", "ROUGE-2 F1 (%)", t1_custom_a["ROUGE-2_F1"], t1_custom_b["ROUGE-2_F1"]),
    build_comparison_row("Tier 1 (NLP - Custom 50)", "ROUGE-L F1 (%)", t1_custom_a["ROUGE-L_F1"], t1_custom_b["ROUGE-L_F1"]),
    build_comparison_row("Tier 1 (NLP - Custom 50)", "BERTScore F1 (%)", t1_custom_a["BERTScore_F1"], t1_custom_b["BERTScore_F1"]),

    # Tier 1: MTS-Dialog Benchmark
    build_comparison_row("Tier 1 (NLP - MTS-Dialog)", "ROUGE-1 F1 (%)", t1_mts_a["ROUGE-1_F1"], t1_mts_b["ROUGE-1_F1"]),
    build_comparison_row("Tier 1 (NLP - MTS-Dialog)", "BERTScore F1 (%)", t1_mts_a["BERTScore_F1"], t1_mts_b["BERTScore_F1"]),

    # Tier 2: Clinical Factuality, Linguistic Shifting & PICO
    build_comparison_row("Tier 2 (Clinical Factuality)", "Medical Entity Recall (%)", t23_custom_a["Entity_Recall"], t23_custom_b["Entity_Recall"]),
    build_comparison_row("Tier 2 (Clinical Factuality)", "Medical Entity Precision (%)", t23_custom_a["Entity_Precision"], t23_custom_b["Entity_Precision"]),
    build_comparison_row("Tier 2 (Clinical Factuality)", "Medical Entity F1-Score (%)", t23_custom_a["Entity_F1"], t23_custom_b["Entity_F1"]),
    build_comparison_row("Tier 2 (Clinical Factuality)", "Linguistic Shift Accuracy (%)", t23_custom_a["Linguistic_Shift_Rate"], t23_custom_b["Linguistic_Shift_Rate"]),
    build_comparison_row("Tier 2 (Clinical Factuality)", "PICO Schema Completeness (%)", t23_custom_a["PICO_Completeness"], t23_custom_b["PICO_Completeness"]),

    # Tier 3: Safety Guardrails & Hallucination Mitigation
    build_comparison_row("Tier 3 (Safety & Guardrails)", "Negative Guardrail Compliance (%) [↑]", t23_custom_a["Guardrail_Compliance"], t23_custom_b["Guardrail_Compliance"]),
    build_comparison_row("Tier 3 (Safety & Guardrails)", "Small-Talk Noise Leakage (%) [↓]", t23_custom_a["Small_Talk_Leakage"], t23_custom_b["Small_Talk_Leakage"]),
    build_comparison_row("Tier 3 (Safety & Guardrails)", "Ungrounded Hallucination Rate (%) [↓]", t23_custom_a["Hallucination_Rate"], t23_custom_b["Hallucination_Rate"])
]

df_summary = pd.DataFrame(summary_rows)
display_cols = [
    "Evaluation Tier", "Metric",
    "Condition A: Before (Zero-Shot)",
    "Condition B: After (Prompt Assembly)",
    "Absolute Delta (After - Before)",
    "Paired t-test (p-value)"
]

# Save to CSV for IEEE Paper
csv_out = os.path.join(OUTPUT_DIR, "ieee_before_vs_after_metrics_table.csv")
df_summary[display_cols].to_csv(csv_out, index=False)

print("=" * 105)
print("MASTER EMPIRICAL RESULTS: CONDITION A (BEFORE) VS. CONDITION B (AFTER - PROMPT ASSEMBLY + PICO)")
print("=" * 105)
display(df_summary[display_cols])
print(f"\n[SAVED] IEEE Results Table exported to: {csv_out}")

## 6. Generating High-Resolution Figures for the IEEE Paper & Presentation
Finally, we generate a high-resolution (300 DPI) grouped bar chart comparing **Condition A (Before)** vs. **Condition B (After)** across key clinical metrics, saved directly to `/kaggle/working/evaluation_results/ieee_before_vs_after_chart.png`.

In [ ]:
# ==============================================================================
# GENERATE 300-DPI IEEE FIGURE (BEFORE VS. AFTER)
# Purpose: Plot a publication-ready bar chart comparing Condition A vs. B
#          across Tier 1, Tier 2, and Tier 3 metrics.
# ==============================================================================

plot_metrics = [
    "ROUGE-L F1 (%)",
    "BERTScore F1 (%)",
    "Medical Entity F1-Score (%)",
    "Linguistic Shift Accuracy (%)",
    "PICO Schema Completeness (%)",
    "Negative Guardrail Compliance (%) [↑]",
    "Small-Talk Noise Leakage (%) [↓]",
    "Ungrounded Hallucination Rate (%) [↓]"
]

# Drop duplicate metric names (keeps Custom 50 BERTScore and drops MTS-Dialog duplicate)
df_plot = (
    df_summary[df_summary["Metric"].isin(plot_metrics)]
    .drop_duplicates(subset=["Metric"], keep="first")
    .copy()
)

x = np.arange(len(df_plot))
width = 0.36

fig, ax = plt.subplots(figsize=(12, 6), dpi=300)
rects1 = ax.bar(x - width/2, df_plot["mean_a_num"], width, label="Condition A: Before (Zero-Shot Baseline)", color="#8c96c6", edgecolor="black")
rects2 = ax.bar(x + width/2, df_plot["mean_b_num"], width, label="Condition B: After (Prompt Assembly + PICO)", color="#00a087", edgecolor="black")

ax.set_ylabel("Score / Rate (%)", fontsize=12, fontweight="bold")
ax.set_title("IBM Granite 3.1 SLM: Before vs. After Modular Prompt Assembly (N=50 Stress-Test Corpus)", fontsize=13, fontweight="bold", pad=15)
ax.set_xticks(x)
ax.set_xticklabels([m.replace(" (%)", "") for m in df_plot["Metric"]], rotation=22, ha="right", fontsize=10, fontweight="bold")
ax.set_ylim(0, 115)
ax.grid(axis="y", linestyle="--", alpha=0.5)
ax.legend(fontsize=11, loc="upper left")

for rect in rects1:
    h = rect.get_height()
    ax.annotate(f"{h:.1f}%", xy=(rect.get_x() + rect.get_width()/2, h), xytext=(0, 3),
                textcoords="offset points", ha="center", va="bottom", fontsize=8.5)
for rect in rects2:
    h = rect.get_height()
    ax.annotate(f"{h:.1f}%", xy=(rect.get_x() + rect.get_width()/2, h), xytext=(0, 3),
                textcoords="offset points", ha="center", va="bottom", fontsize=8.5, fontweight="bold")

plt.tight_layout()
fig_path = os.path.join(OUTPUT_DIR, "ieee_before_vs_after_chart.png")
plt.savefig(fig_path, dpi=300)
plt.show()